In [1]:
# =========================================================
# 14A_final_ensemble_blending.ipynb
# Final Ensemble Blending (Linear, Ridge, Lasso)
# Using FINAL feature engineered datasets
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings("ignore")

# -------------------------------
# Configuration
# -------------------------------
project_root = Path("C:/JupyterProjects/Stock_ML_Project")
data_dir = project_root / "Data" / "Processed" / "enhanced"
results_dir = project_root / "Results"
results_dir.mkdir(parents=True, exist_ok=True)

tickers = {
    "RELIANCE": data_dir / "reliance_final_model_ready.csv",
    "TCS": data_dir / "tcs_final_model_ready.csv",
    "HDFCBANK": data_dir / "hdfcbank_final_model_ready.csv"
}

models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.001)
}

# -------------------------------
# Helper
# -------------------------------
def evaluate(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

# -------------------------------
# Blending Function
# -------------------------------
def blend_predictions(preds_dict, weights=None):
    model_names = list(preds_dict.keys())
    preds_matrix = np.column_stack(list(preds_dict.values()))

    if weights is None:
        weights = np.ones(len(model_names)) / len(model_names)
    weights = np.array(weights) / np.sum(weights)
    blended = np.dot(preds_matrix, weights)
    return blended

# -------------------------------
# Main Loop
# -------------------------------
results = []

for ticker, path in tickers.items():
    print(f"\n=== Processing {ticker} ===")

    if not path.exists():
        print(f"  ⚠️ File missing: {path}")
        continue

    df = pd.read_csv(path)
    print(f"  Loaded: {df.shape}")

    target_col = None
    for c in ["Target_Reg_y", "Target_Reg"]:
        if c in df.columns:
            target_col = c
            break
    if not target_col:
        print(f"  ⚠️ No target column found.")
        continue

    df = df.replace([np.inf, -np.inf], np.nan)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    df_numeric = df[numeric_cols].copy().dropna(axis=1, how='all')
    imputer = SimpleImputer(strategy="mean")
    df_numeric[df_numeric.columns] = imputer.fit_transform(df_numeric)

    y = df_numeric[target_col]
    X = df_numeric.drop(columns=[target_col], errors="ignore")

    X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False, test_size=0.2)

    preds = {}
    scores = {}

    # Train individual models
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        preds[name] = y_pred
        scores[name] = evaluate(y_test, y_pred)
        print(f"  → {name}: RMSE={scores[name]['RMSE']:.3f}, R2={scores[name]['R2']:.3f}")

    # Simple average blend
    blended_simple = blend_predictions(preds)
    metrics_simple = evaluate(y_test, blended_simple)

    # Weighted blend (inverse RMSE as weight)
    inv_rmse = np.array([1 / scores[m]["RMSE"] for m in models])
    blended_weighted = blend_predictions(preds, inv_rmse)
    metrics_weighted = evaluate(y_test, blended_weighted)

    print(f"  ✅ Blend_Simple: RMSE={metrics_simple['RMSE']:.3f}, R2={metrics_simple['R2']:.3f}")
    print(f"  ✅ Blend_Weighted: RMSE={metrics_weighted['RMSE']:.3f}, R2={metrics_weighted['R2']:.3f}")

    # Store all results
    for m in models:
        row = scores[m].copy()
        row.update({"Model": m, "Ticker": ticker})
        results.append(row)

    for name, metric in [("Blend_Simple", metrics_simple), ("Blend_Weighted", metrics_weighted)]:
        metric.update({"Model": name, "Ticker": ticker})
        results.append(metric)

# -------------------------------
# Save Results
# -------------------------------
results_df = pd.DataFrame(results)
save_path = results_dir / "final_ensemble_blending_results.csv"
results_df.to_csv(save_path, index=False)
print(f"\n✅ Final ensemble blending completed. Results saved to: {save_path}")
display(results_df.sort_values(["Ticker", "R2"], ascending=[True, False]))



=== Processing RELIANCE ===
  Loaded: (1460, 54)
  → Linear: RMSE=16.170, R2=0.981
  → Ridge: RMSE=19.843, R2=0.971
  → Lasso: RMSE=16.540, R2=0.980
  ✅ Blend_Simple: RMSE=17.192, R2=0.978
  ✅ Blend_Weighted: RMSE=17.059, R2=0.978

=== Processing TCS ===
  Loaded: (1460, 54)
  → Linear: RMSE=47.054, R2=0.976
  → Ridge: RMSE=60.649, R2=0.961
  → Lasso: RMSE=48.336, R2=0.975
  ✅ Blend_Simple: RMSE=51.312, R2=0.972
  ✅ Blend_Weighted: RMSE=50.688, R2=0.972

=== Processing HDFCBANK ===
  Loaded: (1460, 54)
  → Linear: RMSE=8.750, R2=0.981
  → Ridge: RMSE=10.817, R2=0.971
  → Lasso: RMSE=8.896, R2=0.980
  ✅ Blend_Simple: RMSE=9.200, R2=0.979
  ✅ Blend_Weighted: RMSE=9.129, R2=0.979

✅ Final ensemble blending completed. Results saved to: C:\JupyterProjects\Stock_ML_Project\Results\final_ensemble_blending_results.csv


,RMSE,MAE,R2,Model,Ticker
10,8.750029,6.325625,0.980968,Linear,HDFCBANK
12,8.895989,6.401042,0.980328,Lasso,HDFCBANK
14,9.129022,6.652197,0.979284,Blend_Weighted,HDFCBANK
13,9.199955,6.715336,0.978961,Blend_Simple,HDFCBANK
11,10.816881,8.153522,0.970915,Ridge,HDFCBANK
0,16.170294,11.276267,0.980664,Linear,RELIANCE
2,16.540265,11.798365,0.979770,Lasso,RELIANCE
4,17.058870,12.218939,0.978481,Blend_Weighted,RELIANCE
3,17.191545,12.348971,0.978145,Blend_Simple,RELIANCE
1,19.843013,14.694499,0.970884,Ridge,RELIANCE
